# Notebook 02: Data Cleaning & Engineering

This notebook represents the robust, production-ready pipeline for preparing raw banking data for Exploratory Data Analysis (EDA) and subsequent Machine Learning models.

## Pipeline Objectives
- **Data Integrity:** Gracefully handle missing inputs and anomalies logically without unnecessary row destruction.
- **Feature Augmentation:** Extract sophisticated temporal and monetary behaviors dynamically.
- **Reliability:** Implement defensive programming and strict validation constraints ensuring crash-free, consistent output.
- **Traceability:** High-visibility structured logging for stakeholder interpretability.

### Step 1: Initialization and Data Loading
We load libraries and establish robust pathing. Strict logging is initiated.

In [11]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

print("Starting Data Cleaning & Feature Engineering Pipeline...")

# Simple dictionary to track data quality
data_quality = {
    "rows_initial": 0,
    "rows_final": 0,
    "duplicates_removed": 0,
    "missing_fixed": 0,
    "anomalies_fixed": 0,
    "features_created": 0,
    "columns_dropped": 0
}

# Load the dataset
raw_data_path = '../data/raw/Banking_churn_prediction.csv'
df = pd.read_csv(raw_data_path)

data_quality["rows_initial"] = len(df)
print(f"Initial data loaded. Size: {len(df)} rows, {len(df.columns)} columns.")

Starting Data Cleaning & Feature Engineering Pipeline...
Initial data loaded. Size: 28382 rows, 21 columns.


### Step 2: Column Normalization
Standardizing column names prevents index mismatches during production scoring endpoints.

In [12]:
# Standardize column names to lowercase and replace spaces with underscore
df.columns = df.columns.astype(str).str.strip().str.replace(' ', '_', regex=False).str.lower()
print("Columns standardized.")

Columns standardized.


### Step 3: Duplicate Handling
Identical signals bias ML convergence. We scan and clean these prior to imputation.

In [13]:
# Find and remove duplicates
duplicate_count = df.duplicated().sum()
print(f"Duplicate row scan: {duplicate_count} found.")

if duplicate_count > 0:
    df.drop_duplicates(inplace=True)
    data_quality["duplicates_removed"] = int(duplicate_count)
    print(f"{duplicate_count} Duplicates removed successfully.")
else:
    print("No duplicates detected.")

Duplicate row scan: 0 found.
No duplicates detected.


### Step 4: Datetime Parsing (`last_transaction`)
We safely coerce temporal structures.
> **Business Context:** Missing values (`NaT`) natively exist here indicating account stagnation. They are captured and retained actively as signals rather than imputed.

In [14]:
# Convert transaction date to datetime object
if 'last_transaction' in df.columns:
    df['last_transaction'] = pd.to_datetime(df['last_transaction'], errors='coerce')
    print("`last_transaction` temporal fields parsed successfully.")
else:
    print("`last_transaction` column not found; skipping parsing.")

`last_transaction` temporal fields parsed successfully.


### Step 5: Missing Value Imputation
Structural continuity across numeric arrays and categorical boundaries.
- **Medians (Numeric):** Circumvents skew/distortion from banking heavy-tails (large localized balances).
- **Modes (Categorical):** Maintains the natural probability distributions.
- **Dates:** Explicitly skipped.

In [15]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=['number']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

imputation_logs = []

# Fill missing numeric values with median, or 0 if all are missing
for col in numeric_cols:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        val_median = df[col].median()
        if pd.isna(val_median):
            val_median = 0
        df[col] = df[col].fillna(val_median)
        data_quality["missing_fixed"] += int(missing_count)
        imputation_logs.append(col)

# Fill missing categorical values with mode
for col in categorical_cols:
    missing_count = df[col].isnull().sum()
    if missing_count > 0:
        mode_vals = df[col].mode()
        if len(mode_vals) > 0:
            val_mode = mode_vals[0] 
        else:
            val_mode = "Unknown"
            
        df[col] = df[col].fillna(val_mode)
        data_quality["missing_fixed"] += int(missing_count)
        imputation_logs.append(col)

if len(imputation_logs) > 0:
    print(f"Applied robust imputation logic to {len(imputation_logs)} columns.")
else:
    print("No missing values required imputation.")

Applied robust imputation logic to 4 columns.


### Step 6: Targeted Anomaly Rationalization
Handling impossible constraints securely.
- **Dependents:** Extreme values (< 0 or > 10) are logically treated as faulty data ingestion elements.
- **Age Context:** Elements measuring < 18 are deliberately retained (representative of minor/joint custody account frameworks).

In [16]:
# Fix anomalies in dependents column
if 'dependents' in df.columns:
    # Capturing bad values below 0 or above 10
    faulty_mask = (df['dependents'] > 10) | (df['dependents'] < 0)
    fault_count = faulty_mask.sum()
    
    if fault_count > 0:
        median_dep = df['dependents'].median()
        df.loc[faulty_mask, 'dependents'] = median_dep
        data_quality["anomalies_fixed"] = int(fault_count)
        print(f"Handled {fault_count} dependent anomalies using median substitution.")
    else:
        print("No dependent anomalies detected.")

Handled 5 dependent anomalies using median substitution.


### Step 7: Feature Engineering Pipeline
Enriching predictive bounds organically:
- `is_active` → Binary indicator of recent activity via transaction existence.
- `balance_diff` → Momentum measurement (cash flow trajectory).
- `customer_tenure_years` → Time normalization.
- `age_group` → Expanded binning constraints protecting against upper-edge bounds.

In [17]:
engineered_count = 0

# Create feature for active users
if 'last_transaction' in df.columns:
    df['is_active'] = df['last_transaction'].notna().astype(int)
    engineered_count += 1

# Calculate balance diff
if 'current_balance' in df.columns and 'previous_month_balance' in df.columns:
    if pd.api.types.is_numeric_dtype(df['current_balance']) and pd.api.types.is_numeric_dtype(df['previous_month_balance']):
        df['balance_diff'] = df['current_balance'] - df['previous_month_balance']
        engineered_count += 1

# Calculate customer tenure based on vintage
if 'vintage' in df.columns:
    df['customer_tenure_years'] = (df['vintage'] / 365).round(2)
    engineered_count += 1

# Group age
if 'age' in df.columns:
    # Added include_lowest=True and np.inf to handle all numbers safely
    bins = [0, 18, 30, 45, 60, np.inf]
    labels = ['0-17', '18-29', '30-44', '45-59', '60+']
    df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, include_lowest=True)
    engineered_count += 1

data_quality["features_created"] = engineered_count
print(f"Engineered {engineered_count} new features successfully.")

Engineered 4 new features successfully.


### Step 8: Safe Column Pruning & Data Integrity Guardrails
Applying conditional enforcement of data types and clearing non-predictive variables securely.

In [18]:
columns_dropped = []

# Make city a category if it's not a number
if 'city' in df.columns:
    if pd.api.types.is_numeric_dtype(df['city']):
        print("`city` is numeric. Maintained as regional ID structure.")
    else:
        df['city'] = df['city'].astype('category')
        print("`city` securely cast as categorical array.")

# Drop columns that are no longer needed
drop_targets = ['customer_id', 'branch_code', 'vintage']
for target in drop_targets:
    if target in df.columns:
        df.drop(columns=[target], inplace=True)
        columns_dropped.append(target)

data_quality["columns_dropped"] = len(columns_dropped)
print(f"Removed {len(columns_dropped)} non-predictive/obsolete arrays: {columns_dropped}")

# Create final copy
df_final = df.copy()

`city` is numeric. Maintained as regional ID structure.
Removed 3 non-predictive/obsolete arrays: ['customer_id', 'branch_code', 'vintage']


### Step 9: Defensive Validation Layer
We assert strict compliance checks guaranteeing dataset structure is identical to intended logic frameworks before saving. Bypassing issues is unacceptable.

In [19]:
print("Executing Final Validation Assertions...")

# Check for duplicates again
if df_final.duplicated().sum() == 0:
    print("Assertion Passed: No duplicates.")
else:
    print("Warning: Duplicates persist.")

# Check for unexpected nulls (only allow expected ones like dates)
remaining_nulls = df_final.isnull().sum()
null_cols = remaining_nulls[remaining_nulls > 0].index.tolist()

allowed_nulls = ['last_transaction', 'age_group']
unauthorized_nulls = [nc for nc in null_cols if nc not in allowed_nulls]

if len(unauthorized_nulls) == 0:
    print("Assertion Passed: Missing value limits enforced.")
else:
    print(f"Warning: Unexpected NaNs in {unauthorized_nulls}")

print(f"\n Final shape verification: {df_final.shape}")

Executing Final Validation Assertions...
Assertion Passed: No duplicates.
Assertion Passed: Missing value limits enforced.

 Final shape verification: (28382, 22)


### Step 10: Final Export & Summary
Data Quality Summary is formulated and the dataframe is safely exported to the central processing hub.

In [20]:
# Save clean data 
output_path = '../data/processed/cleaned_banking_churn.csv'
df_final.to_csv(output_path, index=False)

data_quality["rows_final"] = len(df_final)

# Print nice final summary
print("\n=============================================")
print("  PIPELINE DATA QUALITY & EXIT SUMMARY ")
print("=============================================")
print(f"Total Rows Processed: {data_quality['rows_initial']}")
print(f"Final Output Rows: {data_quality['rows_final']}")
print(f"Duplicates Removed: {data_quality['duplicates_removed']}")
print(f"Missing Values Fixed: {data_quality['missing_fixed']}")
print(f"Anomalies Fixed: {data_quality['anomalies_fixed']}")
print(f"Features Engineered: {data_quality['features_created']}")
print(f"Columns Filtered Out: {data_quality['columns_dropped']}")
print("Production Integrity Validation: PASSED")
print(f"Dataset successfully deployed to: {output_path}")
print("=============================================")


  PIPELINE DATA QUALITY & EXIT SUMMARY 
Total Rows Processed: 28382
Final Output Rows: 28382
Duplicates Removed: 0
Missing Values Fixed: 3871
Anomalies Fixed: 5
Features Engineered: 4
Columns Filtered Out: 3
Production Integrity Validation: PASSED
Dataset successfully deployed to: ../data/processed/cleaned_banking_churn.csv
